In [1]:
import optax

In [18]:
schedule = optax.linear_schedule(0.0, 1.0, max(1, 0.1*100))

In [34]:
schedule(15).item()

1.0

In [35]:
# load the tokenizer
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")


/Users/chinmay/Programming/annealedRL/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [66]:
sample_text = "yo can you tell me 2 + 2?"

trace = "ok so first i need to do addition. 2 is 2 and then 2 + 2 is 4. so the answer is 4"

In [61]:
def apply_annealing(tokens: list[int], percent: float) -> list[int]: # only applied to thinking trace
    num_tokens = len(tokens)
    assert percent >= 0.0 and percent <= 1.0, "Annealing percentage must be between 0.0 and 1.0"
    num_annealing_tokens = int(num_tokens * percent)
    annealing_tokens = tokens[:num_annealing_tokens]
    return annealing_tokens


def apply_prompt_template(text: str) -> str:
    return f"""Solve the following math problem step by step. Put your answer inside \\boxed{{}}.
{text}
Remember to put your answer inside \\boxed{{}}."""


def apply_system_prompt_template() -> str:
    return r"""You are a helpful AI assistant.
For every problem, you must reason step-by-step inside <think></think> tags before giving the final answer.
"""


def get_chat_template(system_prompt: bool, text: str) -> list[dict[str, str]]:
    chat = []
    if system_prompt:
        chat.append({"role": "system", "content": apply_system_prompt_template()})
    chat.append({"role": "user", "content": apply_prompt_template(text)})
    return chat

In [50]:
chat_template = (tokenizer.apply_chat_template(
    get_chat_template(True, sample_text), 
    add_generation_prompt=True, 
    enable_thinking=True
, 
    tokenize=False))

trace_tokens = tokenizer.encode(trace, add_special_tokens=False)
annealed_trace = tokenizer.decode(apply_annealing(trace_tokens, 0.5, 1024))

print(chat_template + annealed_trace)


<|im_start|>system
You are a helpful AI assistant.
For every problem, you must reason step-by-step inside <think></think> tags before giving the final answer.
<|im_end|>
<|im_start|>user
Solve the following math problem step by step. Put your answer inside \boxed{}.
yo can you tell me 2 + 2?
Remember to put your answer inside \boxed{}.<|im_end|>
<|im_start|>assistant
ok so first i need to do addition. 2 is 2 and


In [71]:
def prepare_prompt(text: str, trace: str, annealing_percentage: float) -> list[int]:
        assert annealing_percentage is not None, "Annealing percentage must be provided"
        assert annealing_percentage >= 0.0 and annealing_percentage <= 1.0, "Annealing percentage must be between 0.0 and 1.0"

        base = tokenizer.apply_chat_template(
            get_chat_template(True, text),
            add_generation_prompt=True,
            enable_thinking=True,
            tokenize=True,
        )

        if annealing_percentage == 0.0:
            return base

        trace_tokens = tokenizer.encode(trace, add_special_tokens=False)
        annealed_trace = apply_annealing(trace_tokens, annealing_percentage)

        return base + annealed_trace


In [73]:
tokens = prepare_prompt(sample_text, trace, 0.5)

In [74]:
tokenizer.decode(tokens)

'<|im_start|>system\nYou are a helpful AI assistant.\nFor every problem, you must reason step-by-step inside <think></think> tags before giving the final answer.\n<|im_end|>\n<|im_start|>user\nSolve the following math problem step by step. Put your answer inside \\boxed{}.\nyo can you tell me 2 + 2?\nRemember to put your answer inside \\boxed{}.<|im_end|>\n<|im_start|>assistant\nok so first i need to do addition. 2 is 2 and'

In [75]:
print(tokenizer.decode(tokens))

<|im_start|>system
You are a helpful AI assistant.
For every problem, you must reason step-by-step inside <think></think> tags before giving the final answer.
<|im_end|>
<|im_start|>user
Solve the following math problem step by step. Put your answer inside \boxed{}.
yo can you tell me 2 + 2?
Remember to put your answer inside \boxed{}.<|im_end|>
<|im_start|>assistant
ok so first i need to do addition. 2 is 2 and


In [47]:
chat_template = (tokenizer.apply_chat_template(
    get_chat_template(True, sample_text), 
    add_generation_prompt=True, 
    enable_thinking=True
, 
    tokenize=False))
chat_template

'<|im_start|>system\nYou are a helpful AI assistant.\nFor every problem, you must reason step-by-step inside <think></think> tags before giving the final answer.\n<|im_end|>\n<|im_start|>user\nSolve the following math problem step by step. Put your answer inside \\boxed{}.\nyo can you tell me 2 + 2?\nRemember to put your answer inside \\boxed{}.<|im_end|>\n<|im_start|>assistant\n'